# R27-H292 - Type-label reconciliation before identity judging

The single-shot identity judge (H282) detects 98.0% of SAME_AS false merges but over-splits true
cross-type aliases: preservation collapses to 0.29 / 0.00 exactly on the two type-conflict strata
because conflicting extractor type labels (Accessory vs Feature, KGFEntityVersion vs Accessory)
trigger the judge's type-mismatch rule. The labels are extraction noise, not identity evidence.

This notebook tests three reconciliation candidates against the **frozen** 127-pair labels and the
**warm** H282 LLM cache. Only the type-label treatment changes between H282 and each candidate, so
every delta attributes to reconciliation and nothing else.

- **type-blind** - strip all type labels from the judge context; decide on names / descriptions / provenance
- **multi-label union** - present both type labels as jointly valid ("this entity may be both X and Y")
- **cured-ontology arbitration** - one cheap call re-types the pair under the cured ontology's semantic labels, then judge

**Bars (per candidate)**: type-conflict strata TM-preservation >= 0.75 (from 0.29/0.00) AND overall
FM-detection >= 95%. All 127 pairs run per candidate - detection regression on the clean / code_shared
strata is exactly what the second clause guards.

**Substrate**: `reports/r27-precision-labels-frozen-20260708T095229Z.json` (127 pairs, 26 true / 101 false),
`results/r27_precision/llm_cache.json` (warm H282 prompts), pair evidence reproduced byte-exact from
neo4j2 (read-only, `bolt://172.19.0.9:7687`). Endpoint: vLLM localhost:8010, gpt-oss-120b, temp 0,
reasoning_effort=medium, <= 4 concurrent.

## Imports

In [1]:
import os, re, json, time, hashlib, unicodedata, collections, statistics
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from neo4j import GraphDatabase          # read-only reference graph
from openai import OpenAI                # vLLM OpenAI-compatible endpoint
from rich.console import Console
from rich.table import Table
con = Console()
con.print("imports ready")

imports ready

## Configuration

In [2]:
NEO4J_URI  = "bolt://172.19.0.9:7687"          # READ-ONLY reference graph (neo4j2) - MATCH only
NEO4J_AUTH = ("neo4j", "kgfoundry")
LLM_BASE   = "http://localhost:8010/v1"
LLM_MODEL  = "gpt-oss-120b"
TEMP       = 0.0
REASONING_EFFORT = "medium"                     # high is known-broken under tool protocols
MAX_INFLIGHT = 4                                # shared GPU - keep <=4 concurrent
UTC        = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

ROOT        = Path("..").resolve()
FROZEN_PATH = ROOT/"reports/r27-precision-labels-frozen-20260708T095229Z.json"
CKPT_DIR    = ROOT/"results/r27_precision"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
WARM_CACHE  = CKPT_DIR/"llm_cache.json"          # warm H282 prompts (reuse, read-only)
REPORT_PATH = ROOT/f"reports/type-reconciliation-h292-{UTC}.json"

t = Table(title="R27-H292 config", show_header=False)
for k,v in [("neo4j2 (RO)",NEO4J_URI),("LLM",f"{LLM_MODEL} @ {LLM_BASE}"),
            ("temp",TEMP),("effort",REASONING_EFFORT),("max in-flight",MAX_INFLIGHT),
            ("UTC",UTC),("frozen labels",FROZEN_PATH.name),("report",REPORT_PATH.name)]:
    t.add_row(str(k),str(v))
con.print(t)

                           R27-H292 config                           
┌───────────────┬───────────────────────────────────────────────────┐
│ neo4j2 (RO)   │ bolt://172.19.0.9:7687                            │
│ LLM           │ gpt-oss-120b @ http://localhost:8010/v1           │
│ temp          │ 0.0                                               │
│ effort        │ medium                                            │
│ max in-flight │ 4                                                 │
│ UTC           │ 20260708T100715Z                                  │
│ frozen labels │ r27-precision-labels-frozen-20260708T095229Z.json │
│ report        │ type-reconciliation-h292-20260708T100715Z.json    │
└───────────────┴───────────────────────────────────────────────────┘

## Load reference graph (read-only) and reproduce H282 node context

In [3]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
def q(cypher, **p):
    with driver.session() as s:
        return [r.data() for r in s.run(cypher, **p)]

# --- rebuild nodes exactly as the H282 arm did (byte-exact context reproduction) ---
nodes = {}
for r in q("MATCH (e) WHERE e.name IS NOT NULL "
           "RETURN e.id AS id, e.name AS name, labels(e) AS labs, e.description AS desc, "
           "e.source_chunks AS chunks, e.source_documents AS docs"):
    labs = [l for l in r["labs"] if l != "Entity"]
    nodes[r["id"]] = dict(name=r["name"], labs=labs, prim=(labs[0] if labs else "Entity"),
                          desc=r["desc"] or "", chunks=r["chunks"] or [], docs=r["docs"] or [])

CONTENT_EXCLUDE = {"MENTIONED_IN","ABOUT","SIMILAR_TO","SAME_AS","HAD_VERSION"}
adj = collections.defaultdict(list)
for r in q("MATCH (a)-[x]->(b) WHERE a.id IS NOT NULL AND b.id IS NOT NULL "
           "RETURN a.id AS a, type(x) AS t, b.id AS b"):
    if r["t"] not in CONTENT_EXCLUDE:
        adj[r["a"]].append((r["t"], r["b"])); adj[r["b"]].append((r["t"], r["a"]))
con.print(f"named nodes: [cyan]{len(nodes)}[/cyan]  content-adj entities: [cyan]{len(adj)}[/cyan]")

# --- cured ontology vocabulary: semantic primary labels present in the graph, ---
# --- structural / versioning / degenerate labels dropped (they ARE the noise). ---
STRUCTURAL = {"KGFEntityVersion","KGFDocument","Entity","Equipment","Component"}
_lc = collections.Counter(n["prim"] for n in nodes.values())
CURED_VOCAB = [k for k,v in _lc.most_common() if k not in STRUCTURAL and v >= 5]
con.print(f"cured ontology vocab ({len(CURED_VOCAB)} labels): {CURED_VOCAB}")

named nodes: 2825  content-adj entities: 2679

cured ontology vocab (20 labels): ['Accessory', 'Feature', 'CPAPDevice', 'Specification', 'ComfortFeature', 
'ClinicalFeature', 'EventType', 'Software', 'ProductModel', 'OperatingMode', 'Organization', 'ConnectivityDevice', 
'MedicalCondition', 'Standard', 'Manufacturer', 'Condition', 'TestProtocol', 'Person', 'Location', 
'DataStorageDevice']

## Load frozen labels (no relabeling) and attach reproduced context

In [4]:
frozen = json.load(open(FROZEN_PATH))
pairs = []
for p in frozen["pairs"]:
    a, b = p["key"].split("|")
    assert a in nodes and b in nodes, f"missing node for {p['key']}"
    pairs.append({**p, "a": a, "b": b})
ng = collections.Counter(p["gold"] for p in pairs)
con.print(f"frozen pairs: [cyan]{len(pairs)}[/cyan]  "
          f"[green]{ng['MERGE']} MERGE[/green] / [red]{ng['DISTINCT']} DISTINCT[/red]")
cc = collections.Counter(p["cls"] for p in pairs)
con.print("strata:", dict(cc))
# type-conflict strata = the must-fix set (union of the two conflict classes)
TYPE_CONFLICT_STRATA = {"type_conflict","type_conflict+code_shared"}
n_tc_true = sum(1 for p in pairs if p["cls"] in TYPE_CONFLICT_STRATA and p["gold"]=="MERGE")
con.print(f"type-conflict strata: {sum(p['cls'] in TYPE_CONFLICT_STRATA for p in pairs)} pairs, "
          f"[green]{n_tc_true} true merges[/green] (preservation denominator)")

frozen pairs: 127  26 MERGE / 101 DISTINCT

strata:
{'type_conflict+code_shared': 30, 'code_shared': 34, 'type_conflict': 31, 'clean': 32}

type-conflict strata: 61 pairs, 8 true merges (preservation denominator)

## LLM harness - warm-cache reuse + per-candidate disk cache

Same cached temp-0 chat-completion harness as H282. The warm cache (`llm_cache.json`) is read-only and
lets multi-label reuse the byte-identical H282 result on non-conflict pairs (zero new calls). Each
candidate persists its own responses to `results/r27_precision/h292_<candidate>.json`.

In [5]:
client = OpenAI(base_url=LLM_BASE, api_key="x")
warm = json.load(open(WARM_CACHE)) if WARM_CACHE.exists() else {}

def _key(messages, max_tokens, tag):
    return hashlib.sha1((tag+"|"+json.dumps(messages)+f"|{max_tokens}|{REASONING_EFFORT}").encode()).hexdigest()

def llm(messages, cache, cache_path, max_tokens=1024, tag=""):
    """Cached temp-0 completion. Checks candidate cache, then warm cache. Returns (content, tokens)."""
    key = _key(messages, max_tokens, tag)
    if key in cache:      c = cache[key]; return c["content"], c["tokens"]
    if key in warm:       c = warm[key];  cache[key] = c; return c["content"], c["tokens"]
    for attempt in range(4):
        try:
            r = client.chat.completions.create(model=LLM_MODEL, temperature=TEMP, messages=messages,
                                               max_tokens=max_tokens, extra_body={"reasoning_effort": REASONING_EFFORT})
            content = r.choices[0].message.content or ""
            tok = r.usage.total_tokens
            cache[key] = dict(content=content, tokens=tok)
            return content, tok
        except Exception as e:
            if attempt == 3: raise
            time.sleep(2*(attempt+1))

def run_pool(items, fn):
    out = [None]*len(items)
    with ThreadPoolExecutor(max_workers=MAX_INFLIGHT) as ex:
        futs = {ex.submit(fn, it): i for i, it in enumerate(items)}
        for f in as_completed(futs):
            out[futs[f]] = f.result()
    return out

def parse_verdict(text):
    if not text: return None
    m = re.search(r'"?verdict"?\s*[:=]\s*"?(MERGE|DISTINCT|SAME|SAME_AS|YES|NO|DIFFERENT)"?', text, re.I)
    if m:
        v = m.group(1).upper()
        return "MERGE" if v in ("MERGE","SAME","SAME_AS","YES") else "DISTINCT"
    up = text.upper()
    if "DISTINCT" in up or "DIFFERENT" in up: return "DISTINCT"
    if "MERGE" in up or "SAME ENTITY" in up: return "MERGE"
    return None
con.print("harness ready (warm-cache reuse + per-candidate cache)")

harness ready (warm-cache reuse + per-candidate cache)

## Context builders - identical to H282 except the type-label treatment

`SYS_JUDGE` is byte-identical to the H282 arm. `base_ctx` is the H282 context. Each candidate touches
ONLY how type labels appear (or don't) in that context, so every metric delta attributes to reconciliation.

In [6]:
SYS_JUDGE = ("You are an entity-resolution adjudicator for a medical-device knowledge graph. "
    "Decide whether two graph nodes denote the SAME real-world entity (MERGE) or DIFFERENT entities "
    "(DISTINCT). A device is DISTINCT from its accessory; a mask is DISTINCT from a battery even if they "
    "share a code token; sibling model variants are DISTINCT; pure spelling/spacing/case aliases are MERGE. "
    'Reply with ONLY a JSON object: {"verdict":"MERGE"|"DISTINCT","confidence":0-1,"rationale":"<=20 words"}.')

def neigh(x, typed=True, maxn=6):
    out=[]
    for rel, nb in adj.get(x, [])[:maxn*2]:
        if nb in nodes:
            out.append(f"-{rel}-> [{nodes[nb]['prim']}] {nodes[nb]['name']}" if typed
                       else f"-{rel}-> {nodes[nb]['name']}")
        if len(out)>=maxn: break
    return "; ".join(out) or "(none)"

# ----- H282 reference context (byte-exact) -----
def base_ctx(p):
    na, nb = nodes[p["a"]], nodes[p["b"]]
    return (f'A: [{na["prim"]}] "{na["name"]}"\n  desc: {na["desc"][:200] or "(none)"}\n'
            f'  neighbors: {neigh(p["a"])}\n'
            f'B: [{nb["prim"]}] "{nb["name"]}"\n  desc: {nb["desc"][:200] or "(none)"}\n'
            f'  neighbors: {neigh(p["b"])}\n'
            f'shared source docs: {sorted(set(na["docs"]) & set(nb["docs"])) or "(none)"}')

# ----- candidate 1: TYPE-BLIND - strip all type labels (entities + neighbors) -----
def ctx_typeblind(p):
    na, nb = nodes[p["a"]], nodes[p["b"]]
    return (f'A: "{na["name"]}"\n  desc: {na["desc"][:200] or "(none)"}\n'
            f'  neighbors: {neigh(p["a"], typed=False)}\n'
            f'B: "{nb["name"]}"\n  desc: {nb["desc"][:200] or "(none)"}\n'
            f'  neighbors: {neigh(p["b"], typed=False)}\n'
            f'shared source docs: {sorted(set(na["docs"]) & set(nb["docs"])) or "(none)"}')

# ----- candidate 2: MULTI-LABEL UNION - present both labels as jointly valid (conflict pairs only) -----
def ctx_multilabel_note(p):
    # only fires on genuine type conflicts; on ta==tb the context is byte-identical to H282
    return (f'Note: a single real-world entity may legitimately carry BOTH type labels '
            f'[{p["ta"]}] and [{p["tb"]}] at once; a difference in type label alone is NOT '
            f'evidence that the two nodes are distinct.\n\n')

# ----- candidate 3: CURED-ONTOLOGY - context with reconciled types substituted for extractor labels -----
def ctx_cured(p, rta, rtb):
    na, nb = nodes[p["a"]], nodes[p["b"]]
    return (f'A: [{rta}] "{na["name"]}"\n  desc: {na["desc"][:200] or "(none)"}\n'
            f'  neighbors: {neigh(p["a"])}\n'
            f'B: [{rtb}] "{nb["name"]}"\n  desc: {nb["desc"][:200] or "(none)"}\n'
            f'  neighbors: {neigh(p["b"])}\n'
            f'shared source docs: {sorted(set(na["docs"]) & set(nb["docs"])) or "(none)"}')
con.print("context builders ready (H282 base + 3 type-label treatments)")

context builders ready (H282 base + 3 type-label treatments)

## Metrics - per-stratum and the two binding numbers

`metrics` reproduces the H282 metric definitions. The two bars: TM-preservation on the union of the
two type-conflict strata (>= 0.75) and overall FM-detection (>= 95%).

In [7]:
def metrics(sub, verd):
    fm = [(p,v) for p,v in zip(sub,verd) if p["gold"]=="DISTINCT"]
    tm = [(p,v) for p,v in zip(sub,verd) if p["gold"]=="MERGE"]
    fm_det = sum(v=="DISTINCT" for _,v in fm)/len(fm) if fm else None
    tm_pres= sum(v=="MERGE"    for _,v in tm)/len(tm) if tm else None
    losses = [(p["an"],p["bn"],p["gold_lowconf"]) for p,v in tm if v!="MERGE"]
    false_merges = [(p["an"],p["bn"],p["cls"]) for p,v in fm if v=="MERGE"]  # admitted false merges
    return dict(n=len(sub), n_false=len(fm), n_true=len(tm),
                false_merge_detection=fm_det, true_merge_preservation=tm_pres,
                true_merge_losses=losses, n_true_merge_loss=len(losses),
                false_merges_admitted=false_merges, n_false_merge_admitted=len(false_merges),
                unresolved=sum(v is None for v in verd))

def per_class(sub, verd):
    out={}
    for cls in sorted(set(p["cls"] for p in sub)):
        s=[(p,v) for p,v in zip(sub,verd) if p["cls"]==cls]
        out[cls]=metrics([p for p,_ in s],[v for _,v in s])
    return out

def type_conflict_pres(sub, verd):
    s=[(p,v) for p,v in zip(sub,verd) if p["cls"] in TYPE_CONFLICT_STRATA]
    return metrics([p for p,_ in s],[v for _,v in s])["true_merge_preservation"]

def summarize(name, verd, toks):
    M = metrics(pairs, verd)
    M["per_class"]      = per_class(pairs, verd)
    M["type_conflict_strata_preservation"] = type_conflict_pres(pairs, verd)
    M["tokens_per_pair"]= round(statistics.mean(toks),1)
    M["candidate"]      = name
    tc = M["type_conflict_strata_preservation"]; fd = M["false_merge_detection"]
    M["pass_tc_preservation"] = tc >= 0.75
    M["pass_fm_detection"]    = fd >= 0.95
    M["verdict"] = "PASS" if (tc>=0.75 and fd>=0.95) else "FAIL"
    return M
con.print("metrics ready")

metrics ready

## Candidate 1 - TYPE-BLIND judging (predicted winner)

In [8]:
CACHE_TB = CKPT_DIR/"h292_typeblind.json"
cache_tb = json.load(open(CACHE_TB)) if CACHE_TB.exists() else {}
def run_tb(p):
    msgs=[{"role":"system","content":SYS_JUDGE},
          {"role":"user","content":"Judge this pair.\n\n"+ctx_typeblind(p)}]
    content, tok = llm(msgs, cache_tb, CACHE_TB, tag="H292_typeblind")
    return dict(verdict=parse_verdict(content), tokens=tok)
res_tb = run_pool(pairs, run_tb)
json.dump(cache_tb, open(CACHE_TB,"w"))
v_tb = [r["verdict"] for r in res_tb]; tok_tb=[r["tokens"] for r in res_tb]
M_tb = summarize("type-blind", v_tb, tok_tb)
con.print(f"[bold]type-blind[/bold]  type-conflict-strata preservation {M_tb['type_conflict_strata_preservation']:.1%}  "
          f"overall FM-detection {M_tb['false_merge_detection']:.1%}  tokens/pair {M_tb['tokens_per_pair']:.0f}  "
          f"[{'green' if M_tb['verdict']=='PASS' else 'red'}]{M_tb['verdict']}[/]")
if M_tb["false_merges_admitted"]:
    con.print("[red]NEW false merges admitted:[/red]", M_tb["false_merges_admitted"])

type-blind  type-conflict-strata preservation 75.0%  overall FM-detection 95.0%  tokens/pair 480  PASS

NEW false merges admitted:
[
    ('CT2 adult Alice 5', 'CT2 effort sensor', 'code_shared'),
    ('HC230 Product Range', 'HC230 Standard Range', 'type_conflict+code_shared'),
    ('bCPAP prongs', 'CPAP prongs', 'clean'),
    ('Full Face Mask', 'Full-Face Mask', 'type_conflict'),
    ('AirFit F30', 'AirFit F30 for AirMini', 'code_shared')
]

## Candidate 2 - MULTI-LABEL union (both labels jointly valid)

In [9]:
CACHE_ML = CKPT_DIR/"h292_multilabel.json"
cache_ml = json.load(open(CACHE_ML)) if CACHE_ML.exists() else {}
def run_ml(p):
    if p["ta"] == p["tb"]:
        # no type conflict -> context byte-identical to H282; reuse warm result (tag=H282)
        msgs=[{"role":"system","content":SYS_JUDGE},
              {"role":"user","content":"Judge this pair.\n\n"+base_ctx(p)}]
        content, tok = llm(msgs, cache_ml, CACHE_ML, tag="H282")
    else:
        msgs=[{"role":"system","content":SYS_JUDGE},
              {"role":"user","content":"Judge this pair.\n\n"+ctx_multilabel_note(p)+base_ctx(p)}]
        content, tok = llm(msgs, cache_ml, CACHE_ML, tag="H292_multilabel")
    return dict(verdict=parse_verdict(content), tokens=tok)
res_ml = run_pool(pairs, run_ml)
json.dump(cache_ml, open(CACHE_ML,"w"))
v_ml = [r["verdict"] for r in res_ml]; tok_ml=[r["tokens"] for r in res_ml]
M_ml = summarize("multi-label", v_ml, tok_ml)
con.print(f"[bold]multi-label[/bold]  type-conflict-strata preservation {M_ml['type_conflict_strata_preservation']:.1%}  "
          f"overall FM-detection {M_ml['false_merge_detection']:.1%}  tokens/pair {M_ml['tokens_per_pair']:.0f}  "
          f"[{'green' if M_ml['verdict']=='PASS' else 'red'}]{M_ml['verdict']}[/]")
if M_ml["false_merges_admitted"]:
    con.print("[red]NEW false merges admitted:[/red]", M_ml["false_merges_admitted"])

multi-label  type-conflict-strata preservation 62.5%  overall FM-detection 96.0%  tokens/pair 546  FAIL

NEW false merges admitted:
[
    ('CT2 adult Alice 5', 'CT2, pediatric', 'code_shared'),
    ('HC230 Product Range', 'HC230 Standard Range', 'type_conflict+code_shared'),
    ('Full Face Mask', 'Full-Face Mask', 'type_conflict'),
    ('Ultra-Fine Filter Disposable', 'Ultra-fine Filter Disposable 1-pack', 'type_conflict')
]

## Candidate 3 - CURED-ONTOLOGY arbitration

One cheap arbiter call re-types both nodes under the cured ontology's semantic labels (structural /
versioning labels like KGFEntityVersion are excluded from the vocab, so a version-wrapped node is
re-typed to its real semantic type). The judge then sees the reconciled types instead of the extractor
labels. Tokens/pair counts the arbiter call plus the judge call.

In [10]:
CACHE_CU = CKPT_DIR/"h292_cured.json"
cache_cu = json.load(open(CACHE_CU)) if CACHE_CU.exists() else {}
ARB_SYS = ("You assign each entity the single best type from a fixed medical-device ontology. "
    "Choose strictly from the allowed list; pick the label that best fits the entity's real-world "
    'role from its name and description. Reply ONLY as JSON: {"type_a":"<label>","type_b":"<label>"}.')
def arbitrate(p):
    na, nb = nodes[p["a"]], nodes[p["b"]]
    u = (f"Allowed types: {', '.join(CURED_VOCAB)}\n\n"
         f'A name: "{na["name"]}"\n  desc: {na["desc"][:200] or "(none)"}\n'
         f'B name: "{nb["name"]}"\n  desc: {nb["desc"][:200] or "(none)"}\n\n'
         "Return the best allowed type for A and for B.")
    content, tok = llm([{"role":"system","content":ARB_SYS},{"role":"user","content":u}],
                       cache_cu, CACHE_CU, max_tokens=256, tag="H292_cured_arb")
    rta = rtb = None
    try:
        j = json.loads(re.search(r"\{.*\}", content, re.S).group(0))
        rta, rtb = j.get("type_a"), j.get("type_b")
    except Exception: pass
    # fall back to the extractor label if the arbiter returns an out-of-vocab / missing value
    if rta not in CURED_VOCAB: rta = na["prim"]
    if rtb not in CURED_VOCAB: rtb = nb["prim"]
    return rta, rtb, tok
def run_cu(p):
    rta, rtb, atok = arbitrate(p)
    msgs=[{"role":"system","content":SYS_JUDGE},
          {"role":"user","content":"Judge this pair.\n\n"+ctx_cured(p, rta, rtb)}]
    content, jtok = llm(msgs, cache_cu, CACHE_CU, tag="H292_cured_judge")
    return dict(verdict=parse_verdict(content), tokens=atok+jtok,
                rta=rta, rtb=rtb, reconciled=(rta==rtb))
res_cu = run_pool(pairs, run_cu)
json.dump(cache_cu, open(CACHE_CU,"w"))
v_cu = [r["verdict"] for r in res_cu]; tok_cu=[r["tokens"] for r in res_cu]
M_cu = summarize("cured-ontology", v_cu, tok_cu)
n_reconciled = sum(r["reconciled"] for r in res_cu)
n_tc_reconciled = sum(r["reconciled"] for r,p in zip(res_cu,pairs) if p["cls"] in TYPE_CONFLICT_STRATA)
con.print(f"[bold]cured-ontology[/bold]  type-conflict-strata preservation {M_cu['type_conflict_strata_preservation']:.1%}  "
          f"overall FM-detection {M_cu['false_merge_detection']:.1%}  tokens/pair {M_cu['tokens_per_pair']:.0f}  "
          f"[{'green' if M_cu['verdict']=='PASS' else 'red'}]{M_cu['verdict']}[/]")
con.print(f"arbiter collapsed types to equal on {n_reconciled}/127 pairs "
          f"({n_tc_reconciled} within the type-conflict strata)")
if M_cu["false_merges_admitted"]:
    con.print("[red]NEW false merges admitted:[/red]", M_cu["false_merges_admitted"])
M_cu["n_reconciled_equal"]=n_reconciled; M_cu["n_tc_reconciled_equal"]=n_tc_reconciled

cured-ontology  type-conflict-strata preservation 25.0%  overall FM-detection 95.0%  tokens/pair 980  FAIL

arbiter collapsed types to equal on 70/127 pairs (11 within the type-conflict strata)

NEW false merges admitted:
[
    ('CT2 adult Alice 5', 'CT2, pediatric', 'code_shared'),
    ('HC230 Product Range', 'HC230 Standard Range', 'type_conflict+code_shared'),
    ('bCPAP prongs', 'CPAP prongs', 'clean'),
    ('Full Face Mask', 'Full-Face Mask', 'type_conflict'),
    ('AirFit F30', 'AirFit F30 for AirMini', 'code_shared')
]

## Per-stratum comparison table (all candidates vs H282 baseline)

In [11]:
H282 = json.load(open(CKPT_DIR/"h282.json"))
STRATA = ["clean","code_shared","type_conflict","type_conflict+code_shared"]
def pc(M, cls, field): return M["per_class"][cls][field]

tb = Table(title="TM-preservation by stratum (bar: type-conflict strata union >= 0.75)")
tb.add_column("stratum"); tb.add_column("n_true", justify="right")
for c in ["H282","type-blind","multi-label","cured"]: tb.add_column(c, justify="right")
for cls in STRATA:
    nt = H282["per_class"][cls]["n_true"]
    def cell(M):
        v = pc(M,cls,"true_merge_preservation"); return "-" if v is None else f"{v:.0%}"
    tb.add_row(cls, str(nt),
        (lambda v: "-" if v is None else f"{v:.0%}")(H282["per_class"][cls]["true_merge_preservation"]),
        cell(M_tb), cell(M_ml), cell(M_cu))
def tc_union(M):
    v=M.get("type_conflict_strata_preservation"); return "-" if v is None else f"{v:.0%}"
tb.add_row("[bold]TC strata union[/bold]", str(n_tc_true),
           "25%", tc_union(M_tb), tc_union(M_ml), tc_union(M_cu))
con.print(tb)

tb2 = Table(title="Overall metrics (bar: FM-detection >= 95%)")
tb2.add_column("metric")
for c in ["H282","type-blind","multi-label","cured"]: tb2.add_column(c, justify="right")
tb2.add_row("FM-detection (all 127)", f"{H282['false_merge_detection']:.1%}",
            f"{M_tb['false_merge_detection']:.1%}", f"{M_ml['false_merge_detection']:.1%}", f"{M_cu['false_merge_detection']:.1%}")
tb2.add_row("TC-strata TM-preservation", "25.0%",
            f"{M_tb['type_conflict_strata_preservation']:.1%}", f"{M_ml['type_conflict_strata_preservation']:.1%}", f"{M_cu['type_conflict_strata_preservation']:.1%}")
tb2.add_row("overall TM-preservation", f"{H282['true_merge_preservation']:.1%}",
            f"{M_tb['true_merge_preservation']:.1%}", f"{M_ml['true_merge_preservation']:.1%}", f"{M_cu['true_merge_preservation']:.1%}")
tb2.add_row("new false merges admitted", "0",
            str(M_tb['n_false_merge_admitted']), str(M_ml['n_false_merge_admitted']), str(M_cu['n_false_merge_admitted']))
tb2.add_row("tokens/pair", f"{H282.get('tokens_per_pair',0):.0f}",
            f"{M_tb['tokens_per_pair']:.0f}", f"{M_ml['tokens_per_pair']:.0f}", f"{M_cu['tokens_per_pair']:.0f}")
tb2.add_row("VERDICT vs bars", "baseline",
            M_tb['verdict'], M_ml['verdict'], M_cu['verdict'])
con.print(tb2)

      TM-preservation by stratum (bar: type-conflict strata union >= 0.75)      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┓
┃ stratum                   ┃ n_true ┃ H282 ┃ type-blind ┃ multi-label ┃ cured ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━┩
│ clean                     │     13 │  92% │        92% │         92% │  100% │
│ code_shared               │      5 │  80% │        80% │         80% │   80% │
│ type_conflict             │      7 │  29% │        86% │         71% │   29% │
│ type_conflict+code_shared │      1 │   0% │         0% │          0% │    0% │
│ TC strata union           │      8 │  25% │        75% │         62% │   25% │
└───────────────────────────┴────────┴──────┴────────────┴─────────────┴───────┘

                Overall metrics (bar: FM-detection >= 95%)                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┓
┃ metric                    ┃     H282 ┃ type-blind ┃ multi-label ┃ cured ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━┩
│ FM-detection (all 127)    │    98.0% │      95.0% │       96.0% │ 95.0% │
│ TC-strata TM-preservation │    25.0% │      75.0% │       62.5% │ 25.0% │
│ overall TM-preservation   │    69.2% │      84.6% │       80.8% │ 73.1% │
│ new false merges admitted │        0 │          5 │           4 │     5 │
│ tokens/pair               │      513 │        480 │         546 │   980 │
│ VERDICT vs bars           │ baseline │       PASS │        FAIL │  FAIL │
└───────────────────────────┴──────────┴────────────┴─────────────┴───────┘

## Assemble report

In [12]:
report = dict(
    round="R27-H292", utc=UTC,
    hypothesis="type-label reconciliation lifts type-conflict TM-preservation from 0.25 to >=0.75 "
               "while overall FM-detection stays >=95%",
    bars=dict(type_conflict_strata_preservation=0.75, overall_fm_detection=0.95),
    substrate=dict(frozen_labels=FROZEN_PATH.name, warm_cache="llm_cache.json",
                   n_pairs=len(pairs), n_true=ng["MERGE"], n_false=ng["DISTINCT"],
                   type_conflict_strata_true=n_tc_true),
    endpoint=dict(model=LLM_MODEL, base=LLM_BASE, temp=TEMP, reasoning_effort=REASONING_EFFORT),
    baseline_h282=dict(false_merge_detection=H282["false_merge_detection"],
                       true_merge_preservation=H282["true_merge_preservation"],
                       type_conflict_strata_preservation=0.25),
    candidates={M["candidate"]: M for M in (M_tb, M_ml, M_cu)},
)
passing = [M["candidate"] for M in (M_tb,M_ml,M_cu) if M["verdict"]=="PASS"]
report["passing_candidates"] = passing
report["ships_into_h290"] = (min(((M["tokens_per_pair"], M["candidate"]) for M in (M_tb,M_ml,M_cu)
                                  if M["verdict"]=="PASS"), default=(None,None))[1])
json.dump(report, open(REPORT_PATH,"w"), indent=1)
con.print(f"[bold green]report -> {REPORT_PATH.name}[/bold green]")
con.print(f"passing candidates: {passing or 'NONE'}   ships into H290: {report['ships_into_h290']}")
driver.close()

report -> type-reconciliation-h292-20260708T100715Z.json

passing candidates: ['type-blind']   ships into H290: type-blind

## Verdicts

In [13]:
vt = Table(title="R27-H292 verdicts vs bars")
vt.add_column("candidate"); vt.add_column("TC-strata pres", justify="right")
vt.add_column("FM-det", justify="right"); vt.add_column("new FM", justify="right")
vt.add_column("tok/pair", justify="right"); vt.add_column("verdict")
for M in (M_tb, M_ml, M_cu):
    vt.add_row(M["candidate"], f"{M['type_conflict_strata_preservation']:.0%}",
               f"{M['false_merge_detection']:.1%}", str(M['n_false_merge_admitted']),
               f"{M['tokens_per_pair']:.0f}",
               f"[{'green' if M['verdict']=='PASS' else 'red'}]{M['verdict']}[/]")
con.print(vt)
con.print(f"\n[bold]Bars[/bold]: type-conflict-strata TM-preservation >= 75% (from 25%) "
          f"AND overall FM-detection >= 95%")

                        R27-H292 verdicts vs bars                         
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃ candidate      ┃ TC-strata pres ┃ FM-det ┃ new FM ┃ tok/pair ┃ verdict ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│ type-blind     │            75% │  95.0% │      5 │      480 │ PASS    │
│ multi-label    │            62% │  96.0% │      4 │      546 │ FAIL    │
│ cured-ontology │            25% │  95.0% │      5 │      980 │ FAIL    │
└────────────────┴────────────────┴────────┴────────┴──────────┴─────────┘

Bars: type-conflict-strata TM-preservation >= 75% (from 25%) AND overall FM-detection >= 95%